# Pricing and Hedging of European Options

A binomial-tree (Cox-Ross-Rubinstein) implementation of European option pricing and dynamic hedging, with the risk-neutral theory it rests on.

## Contents
1. [Risk-neutral probability](#1-risk-neutral-probability)
2. [Closed-form price](#2-closed-form-price)
3. [Direct pricer](#3-direct-pricer)
4. [Backward-induction pricer](#4-backward-induction-pricer)
5. [Dynamic hedging](#5-dynamic-hedging)


In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt


## 1. Risk-neutral probability

Discrete time grid $t_0 = 0 < t_1 < \dots < t_N = T$, with $\delta = T/N$.

- Riskless asset: $(1+r_N)^i$ at time $t_i$.
- Risky asset: $S^{(N)}_{t_i} = T^{(N)}_i \, S^{(N)}_{t_{i-1}}$, where $T^{(N)}_i \in \{1+h_N,\ 1+b_N\}$, i.i.d., with $b_N < r_N < h_N$.
- Discounted price: $\tilde S^{(N)}_{t_i} = S^{(N)}_{t_i}/(1+r_N)^i$.

The risk-neutral probability $Q$ is the one under which $(\tilde S^{(N)}_{t_i})_i$ is a martingale. It is characterized by $E_Q[T_1^{(N)}] = 1+r_N$, giving

$$q_N = \frac{r_N - b_N}{h_N - b_N}$$


## 2. Closed-form price

The option paying $f(S_{t_N})$ has discounted risk-neutral price $Prix^{(N)} = \dfrac{1}{(1+r_N)^N}E_Q[f(S^{(N)}_{t_N})]$. Summing over the number of up-moves $k$ among the $N$ steps:

$$Prix^{(N)} = \frac{1}{(1+r_N)^N}\sum_{k=0}^{N} \binom{N}{k} q_N^k (1-q_N)^{N-k}\, f\!\left(s(1+h_N)^k(1+b_N)^{N-k}\right)$$


## 3. Direct pricer

Direct evaluation of the closed-form sum.


In [ ]:
def price1(N, rN, hN, bN, s, f):
    """Binomial price via direct summation over the number of up-moves.

    Parameters
    ----------
    N : int      number of time steps
    rN : float   risk-free rate per step
    hN : float   up factor per step
    bN : float   down factor per step
    s : float    initial spot price
    f : callable payoff function

    Returns
    -------
    float : option price
    """
    qN = (rN - bN) / (hN - bN)
    total = 0.0
    for k in range(N + 1):
        ST = s * (1 + hN)**k * (1 + bN)**(N - k)
        proba = math.comb(N, k) * qN**k * (1 - qN)**(N - k)
        total += proba * f(ST)
    return total / (1 + rN)**N


In [ ]:
call = lambda x: max(x - 100, 0)
price1(20, 0.01, 0.05, -0.05, 100, call)


## 4. Backward-induction pricer

Walks the recombining tree backward from maturity:

$$v_N(S_{t_N}) = f(S_{t_N}), \qquad v_k(S_{t_k}) = \frac{1}{1+r_N}\Big(q_N\, v_{k+1}(\text{up}) + (1-q_N)\, v_{k+1}(\text{down})\Big)$$

Returns the array of option values, collapsing to the price at $t_0$.


In [ ]:
def price2(N, rN, hN, bN, s, f):
    """Binomial price via backward induction on the recombining tree."""
    qN = (rN - bN) / (hN - bN)
    prices = np.zeros(N + 1)
    for k in range(N + 1):
        prices[k] = s * (1 + hN)**k * (1 + bN)**(N - k)
    V = [f(x) for x in prices]
    for k in range(N - 1, -1, -1):
        for i in range(k + 1):
            V[i] = (1 / (1 + rN)) * (qN * V[i + 1] + (1 - qN) * V[i])
    return V


In [ ]:
call = lambda x: max(x - 100, 0)
price2(3, 0.01, 0.05, -0.05, 100, call)[0]


## 5. Dynamic hedging

At a node at date $t_{k-1}$ with asset price $x$, the replicating portfolio $(\alpha_{k-1}, \beta_{k-1})$ holds $\alpha_{k-1}$ units of the risky asset and $\beta_{k-1}$ units of the riskless asset. Matching the option value $v_k$ in both the up and down states at $t_k$ gives a $2\times 2$ linear system with solution

$$\alpha_{k-1}(x) = \frac{v_k\big((1+h_N)x\big) - v_k\big((1+b_N)x\big)}{x(h_N - b_N)} \qquad \text{(Delta)}$$

$$\beta_{k-1}(x) = \frac{(1+h_N)\,v_k\big((1+b_N)x\big) - (1+b_N)\,v_k\big((1+h_N)x\big)}{(1+r_N)^k(h_N - b_N)}$$

The terminal step is the special case $v_N = f$. In tree-index terms, at node $(k-1, j)$ with $x = s(1+h_N)^j(1+b_N)^{k-1-j}$, the up child is `V[j+1]` and the down child is `V[j]`.


In [ ]:
def hedge(N, rN, hN, bN, s, f):
    """Replicating strategy (alpha, beta) at every node before maturity.

    Returns
    -------
    alpha, beta : lists of lists, indexed by step then node.
    """
    alpha = []
    beta = []
    for k in range(1, N + 1):
        alpha_k = []
        beta_k = []
        for j in range(k):
            x = s * (1 + hN)**j * (1 + bN)**(k - 1 - j)
            v_up = price2(N - k, rN, hN, bN, (1 + hN) * x, f)[0]
            v_down = price2(N - k, rN, hN, bN, (1 + bN) * x, f)[0]
            a = (v_up - v_down) / (x * (hN - bN))
            b = ((1 + hN) * v_down - (1 + bN) * v_up) / ((1 + rN)**k * (hN - bN))
            alpha_k.append(a)
            beta_k.append(b)
        alpha.append(alpha_k)
        beta.append(beta_k)
    return alpha, beta


In [ ]:
call = lambda x: max(x - 100, 0)
hedge(2, 0.01, 0.05, -0.05, 100, call)
